In [ ]:
import os
import time

import ipywidgets as widgets
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, clear_output, display
from matplotlib.animation import FuncAnimation
from mplsoccer import Pitch
from scipy.spatial import ConvexHull, Voronoi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

matplotlib.rcParams['animation.embed_limit'] = 100.0

# 1. Carga, limpia y enriquece los datos con táctica.


En esta celda establecemos los cimientos del análisis. Realizamos tres operaciones críticas para transformar datos crudos en información táctica:

1.  **Carga de Datos:** Leemos los archivos de tracking (posiciones a 25Hz) y eventos de Metrica Sports.
2.  **Cálculo de KPIs Geométricos:** Transformamos las coordenadas crudas ($X, Y$) de los 11 jugadores en métricas de equipo usando álgebra matricial (Numpy):
    * **Centroide:** El punto de gravedad del equipo (promedio de posiciones). Nos indica la **Altura del Bloque**.
    * **Área (Spread):** El tamaño del rectángulo que ocupan los jugadores (Max X/Y - Min X/Y). Nos indica la **Compacidad**.
3.  **Creación de la `master_table`:** Cruzamos estas métricas con los eventos para que cada acción (pase, tiro) tenga un contexto táctico asociado (ej: *"Pase realizado en Bloque Medio Muy Compacto"*).

In [ ]:

print("Cargando datos...")
tracking_home = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv', header=2)
tracking_away = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv', header=2)
events = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawEventsData.csv')

def calculate_team_kpis(tracking_df, team_name):
    player_cols = [c for c in tracking_df.columns if c.startswith('Player') and 'Unnamed' not in c]
    
    # Crear array 3D: (Frames, Jugadores, 2 Coordenadas)
    coords = []
    for col in player_cols:
        x = tracking_df[col].values
        # La columna siguiente es Y
        y_idx = tracking_df.columns.get_loc(col) + 1
        y = tracking_df.iloc[:, y_idx].values
        coords.append(np.stack([x, y], axis=1))
    
    all_pos = np.stack(coords, axis=1) # Shape: (n_frames, n_players, 2)
    

    centroid = np.nanmean(all_pos, axis=1) 
    max_pos = np.nanmax(all_pos, axis=1)
    min_pos = np.nanmin(all_pos, axis=1)
    spread = max_pos - min_pos 
    area = spread[:, 0] * spread[:, 1]
    
    return pd.DataFrame({
        'Frame': tracking_df['Frame'],
        f'{team_name}_Centroid_X': centroid[:, 0],
        f'{team_name}_Centroid_Y': centroid[:, 1],
        f'{team_name}_Area': area
    })

# 3. Procesar Tracking
print("⚙️ Calculando KPIs Tácticos...")
kpis_home = calculate_team_kpis(tracking_home, 'Home')
kpis_away = calculate_team_kpis(tracking_away, 'Away')
tracking_enriched = pd.merge(kpis_home, kpis_away, on='Frame')

# 4. Unir con Eventos (Master Table)
master_table = pd.merge(events, tracking_enriched, left_on='Start Frame', right_on='Frame', how='left')

# 5. Traductor Táctico (Texto)
def generate_tactical_text(row):
    # Altura del Bloque (Basado en Home)
    cx = row['Home_Centroid_X']
    if pd.isna(cx): return "Datos no disponibles"
    
    if cx < 0.35: bloque = "Bloque Bajo"
    elif cx < 0.65: bloque = "Bloque Medio"
    else: bloque = "Bloque Alto"
    
    # Compacidad
    area = row['Home_Area']
    if area < 0.12: compacidad = "Muy Compacto"
    elif area < 0.20: compacidad = "Compacto"
    else: compacidad = "Disperso"
    
    return f"{bloque} {compacidad}"

master_table['Tactical_Situation'] = master_table.apply(generate_tactical_text, axis=1)
print("✅ Datos procesados correctamente.")

# 2.  Motor de Visualización de Eventos (Replay Táctico)


Esta función, `animate_play_safe`, actúa como el **reproductor de vídeo** del sistema. Su objetivo es validar visualmente lo que dicen los datos, permitiendo al analista ver "qué pasó realmente" en una jugada específica.

**Funcionamiento Técnico:**
1.  **Slicing Temporal:** Recibe un ID de evento y recorta los DataFrames de tracking gigantes (que tienen todo el partido) para aislar solo los segundos exactos de esa acción, añadiendo un pequeño margen de contexto (1.5s).
2.  **Renderizado Dinámico:** Utiliza la librería `mplsoccer` para dibujar el campo y `FuncAnimation` de Matplotlib para actualizar la posición $(x,y)$ de los 22 jugadores y el balón frame a frame.
3.  **Conversión de Escala:** Transforma en tiempo real las coordenadas normalizadas del dataset (0 a 1) a metros reales (105m $\times$ 68m) para su correcta representación en el gráfico.

In [ ]:

def animate_play_safe(event_id):
    try:
        event = master_table.iloc[event_id]
        start_frame = event['Start Frame']
        end_frame = event['End Frame'] + 40 # 1.5s extra
    except:
        return None
    
    # 2. Slice
    t_home_slice = tracking_home[(tracking_home['Frame'] >= start_frame) & (tracking_home['Frame'] <= end_frame)]
    t_away_slice = tracking_away[(tracking_away['Frame'] >= start_frame) & (tracking_away['Frame'] <= end_frame)]
    
    if t_home_slice.empty: return None

    # 3. Setup Pitch
    pitch = Pitch(pitch_type='custom', pitch_length=105, pitch_width=68, 
                  pitch_color='#195e26', line_color='white')
    fig, ax = pitch.draw(figsize=(10, 6))
    
    scat_home = pitch.scatter([], [], ax=ax, c='red', s=100, edgecolors='white', label='Local', zorder=3)
    scat_away = pitch.scatter([], [], ax=ax, c='blue', s=100, edgecolors='white', label='Visitante', zorder=3)
    scat_ball = pitch.scatter([], [], ax=ax, c='white', s=80, edgecolors='black', zorder=4)
    title = ax.text(52.5, 72, "", ha='center', fontsize=12, fontweight='bold', color='black')

    def get_frame_coords(row, df_cols):
        xs, ys = [], []
        for col in df_cols:
            if str(col).startswith('Player') and 'Unnamed' not in str(col):
                idx = df_cols.get_loc(col)
                val_x = row.iloc[idx]; val_y = row.iloc[idx+1]
                if not pd.isna(val_x):
                    xs.append(val_x * 105); ys.append(val_y * 68)
        return xs, ys

    def update(i):
        if i >= len(t_home_slice) or i >= len(t_away_slice): return scat_home, scat_away, scat_ball, title

        h_row = t_home_slice.iloc[i]; a_row = t_away_slice.iloc[i]
        hx, hy = get_frame_coords(h_row, tracking_home.columns)
        ax_x, ax_y = get_frame_coords(a_row, tracking_away.columns)
        
        bx, by = [], []
        if 'Ball' in tracking_home.columns:
            b_idx = tracking_home.columns.get_loc('Ball')
            val_bx = h_row.iloc[b_idx]; val_by = h_row.iloc[b_idx+1]
            if not pd.isna(val_bx): bx, by = [val_bx * 105], [val_by * 68]

        scat_home.set_offsets(np.c_[hx, hy])
        scat_away.set_offsets(np.c_[ax_x, ax_y])
        if bx: scat_ball.set_offsets(np.c_[bx, by])
        else: scat_ball.set_offsets(np.zeros((0, 2)))

        desc = event.get('Tactical_Situation', '')
        title.set_text(f"Frame: {h_row['Frame']} | {event['Type']} | {desc}")
        return scat_home, scat_away, scat_ball, title

    anim = FuncAnimation(fig, update, frames=len(t_home_slice), interval=40, blit=True)
    plt.close()
    return HTML(anim.to_jshtml())

In [ ]:
animate_play_safe(event_id=5)

# 3. Interfaz de Video-Análisis (Instant Replay)

Hasta ahora hemos construido motores de cálculo y visualización, pero necesitamos una forma humana de interactuar con ellos. Esta celda crea un **Dashboard Interactivo de Eventos**.

**Funcionalidad:**
1.  **Filtrado Inteligente:** Seleccionamos solo los eventos de mayor relevancia (Pases, Tiros, Goles, ABP) de la `master_table` para no saturar al usuario con miles de registros irrelevantes.
2.  **Menú de Navegación:** Generamos un desplegable dinámico que muestra el minuto, tipo de evento y el contexto táctico calculado previamente (ej: *"Bloque Bajo Muy Compacto"*).
3.  **Visualización On-Demand:** Al seleccionar una jugada, el sistema llama al motor de animación (`animate_play_safe`) y renderiza el clip específico en tiempo real, acompañándolo de sus métricas clave.

Es la herramienta definitiva para que el analista valide sus hipótesis viendo la "realidad" detrás del dato.

## no elegir unicamnete los 200 primeros eventos, sino todos los eventos del equipo local y visitante (adaptarlo para que funcione con el partido completo)

In [ ]:



opciones = []
# Limitamos a los primeros 200 eventos con sentido para que cargue rápido
mask = master_table['Type'].isin(['PASS', 'SHOT', 'GOAL', 'SET PIECE'])
df_filt = master_table[mask].head(200)

for idx, row in df_filt.iterrows():
    m = int(row['Start Time [s]'] // 60)
    s = int(row['Start Time [s]'] % 60)
    sit = row.get('Tactical_Situation', 'N/A')
    label = f"ID {idx}: {row['Type']} ({row['Team']}) | {sit} ({m:02d}'{s:02d}'')"
    opciones.append((label, idx))

dropdown = widgets.Dropdown(options=opciones, description='⚽ Evento:', layout={'width': '95%'})
output_plot = widgets.Output()

# 2. Lógica
def al_cambiar(change):
    if change['type'] == 'change' and change['name'] == 'value':
        eid = change['new']
        with output_plot:
            clear_output(wait=True)
            print(f"🎬 Generando repetición para Evento {eid}...")
            video = animate_play_safe(eid)
            if video: display(video)

            # Info extra
            d = master_table.iloc[eid]
            print(f"\n📊 ANÁLISIS: {d['Type']} de {d['From']} ({d['Team']})")
            print(f"🧠 Contexto Táctico: {d.get('Tactical_Situation', 'N/A')}")

dropdown.observe(al_cambiar)

print("👇 Analista de Video Táctico:")
display(dropdown, output_plot)

# Código Unificado: animate_pro_phase (con modo Voronoi opcional)

# 4. Motor de Fases de Juego (Algoritmo de Posesión)

Esta celda contiene el algoritmo heurístico que convierte eventos individuales (pases, conducciones, faltas) en **Unidades de Posesión**. Es fundamental para pasar del análisis estadístico simple al análisis táctico.

**Lógica del Algoritmo:**
1.  **Smart Owner (Dueño Inteligente):** Recorre los eventos secuencialmente para determinar quién tiene el control real.
    * *Confirman posesión:* Pases, Tiros, Regates ganados, Balón Parado.
    * *No cambian posesión:* Interrupciones, tarjetas o faltas recibidas (se mantiene el dueño anterior).
2.  **Segmentación de Fases:** Crea un nuevo `Phase_ID` cada vez que el balón cambia de equipo o el juego se reinicia (ABP).
3.  **Enriquecimiento Táctico:**
    * **Zonas Dinámicas:** Normaliza las coordenadas según la dirección de ataque (Periodo 1 vs 2) para clasificar el inicio/fin en *Iniciación, Creación o Finalización*.
    * **Contexto y Resultado:** Etiqueta automáticamente si la jugada fue una *Recuperación* o *ABP*, y si terminó en *Gol, Tiro o Pérdida*.

In [ ]:


def build_pro_tactical_phases(events_df):
    # Trabajar sobre una copia para no alterar el original
    df = events_df.copy()
    
    # ---------------------------------------------------------
    # 1. LÓGICA DE POSESIÓN (Algoritmo de Dueño)
    # ---------------------------------------------------------
    possession_owners = []
    # Inicializar con el primer equipo que hace algo
    current_owner = df.iloc[0]['Team']
    
    # Eventos que NO cambian la posesión (Interrupciones)
    interruptions = ['FAULT RECEIVED', 'CARD', 'BALL OUT', 'CHALLENGE', 'RECOVERY']
    
    for idx, row in df.iterrows():
        evt_type = row['Type']
        evt_sub = str(row['Subtype'])
        evt_team = row['Team']
        
        # Un regate ganado CONFIRMA la posesión
        if evt_type == 'DRIBBLE' and 'WON' in evt_sub:
            current_owner = evt_team
        # Balón parado INICIA posesión
        elif evt_type == 'SET PIECE':
            current_owner = evt_team
        # Acciones activas CONFIRMAN posesión
        elif evt_type in ['PASS', 'SHOT', 'TOQUE']:
            current_owner = evt_team
        # Si es interrupción, mantenemos el dueño anterior
        elif evt_type in interruptions:
            pass 
        
        possession_owners.append(current_owner)
    
    df['Smart_Owner'] = possession_owners
    
    # Detectar CAMBIO DE FASE:
    # 1. Cambia el equipo dueño.
    # 2. Ocurre un Balón Parado (Set Piece) -> Reinicia jugada.
    df['New_Phase'] = (df['Smart_Owner'] != df['Smart_Owner'].shift()) | (df['Type'] == 'SET PIECE')
    df['Phase_ID'] = df['New_Phase'].cumsum()
    
    # ---------------------------------------------------------
    # 2. AGREGACIÓN (Micro-Ciclos)
    # ---------------------------------------------------------
    # Agrupamos los eventos por Phase_ID para sacar métricas de la jugada completa
    phases = df.groupby('Phase_ID').agg(
        Team=('Smart_Owner', 'first'),
        Period=('Period', 'first'),
        Start_Time=('Start Time [s]', 'min'),
        Duration=('Start Time [s]', lambda x: x.max() - x.min()),
        Event_Count=('Type', 'count'),
        # IMPORTANTE: Usamos nombres con guion bajo para estandarizar
        Start_Frame=('Start Frame', 'min'), 
        End_Frame=('End Frame', 'max'),     
        Start_X=('Start X', 'first'), 
        End_X=('End X', 'last'),    
        Events_List=('Type', list),
        Subtypes_List=('Subtype', list),
        Start_Type=('Type', 'first'),
        Start_Subtype=('Subtype', 'first')
    ).reset_index()
    
    # ---------------------------------------------------------
    # 3. ENRIQUECIMIENTO TÁCTICO (Zonas y Resultado)
    # ---------------------------------------------------------
    
    # Función de Zona (Normalizada 0-1)
    def get_zone(x_coord, period, team):
        # Lógica Metrica: Home ataca a 1 en P1, a 0 en P2
        attack_dir = 1 
        if (team == 'Home' and period == 2) or (team == 'Away' and period == 1):
            attack_dir = -1 
            
        # Normalizar X relativo a "Mi Portería" (0) -> "Rival" (1)
        rel_x = x_coord if attack_dir == 1 else (1.0 - x_coord)
            
        if rel_x < 0.35: return "Iniciación"
        if rel_x < 0.65: return "Creación"
        return "Finalización"

    # Aplicar Zonas
    phases['Start_Zone'] = phases.apply(lambda r: get_zone(r['Start_X'], r['Period'], r['Team']), axis=1)
    phases['End_Zone'] = phases.apply(lambda r: get_zone(r['End_X'], r['Period'], r['Team']), axis=1)
    
    # Definir Contexto (Cómo empezó)
    def define_context(row):
        if row['Start_Type'] == 'SET PIECE': return 'ABP'
        if 'RECOVERY' in row['Events_List']: return 'Recuperación'
        return 'Juego Abierto'
    phases['Context'] = phases.apply(define_context, axis=1)
    
    # Definir Resultado (Outcome)
    def define_outcome(row):
        evs = row['Events_List']
        subs = [str(x) for x in row['Subtypes_List']]
        
        if 'SHOT' in evs:
            if any('GOAL' in s for s in subs): return 'GOL'
            return 'Tiro'
        if 'BALL LOST' in evs: return 'Pérdida'
        if 'BALL OUT' in evs: return 'Fuera'
        return 'Posesión'

    phases['Outcome'] = phases.apply(define_outcome, axis=1)
    
    # Filtrar jugadas "basura" (menos de 1 segundo)
    return phases[phases['Duration'] > 1.0].copy()

# --- EJECUCIÓN PRINCIPAL ---
print("⚙️ Generando Fases Tácticas (pro_phases)...")

# Usamos 'master_table' si existe (viene de la celda 2), si no, usamos 'events' crudo
if 'master_table' in locals():
    pro_phases = build_pro_tactical_phases(master_table)
else:
    print("⚠️ 'master_table' no encontrado, usando 'events' original.")
    pro_phases = build_pro_tactical_phases(events)

print(f"✅ Variable 'pro_phases' definida con {len(pro_phases)} jugadas.")
print("Columnas:", pro_phases.columns.tolist())

# Motor de Animación Táctica (Visualización Micro)

Esta celda define la función `animate_pro_phase`, la herramienta de visualización más avanzada del sistema. Su objetivo es convertir los datos abstractos de una "Fase" en un video táctico comprensible.

**Capas de Visualización:**
1.  **Capa Física:** Renderiza la posición exacta ($x, y$) de los 22 jugadores y el balón a 25 frames por segundo, sincronizando los datos de tracking de ambos equipos.
2.  **Capa Táctica (Voronoi):** Implementa un algoritmo de geometría computacional que divide el campo en regiones según qué jugador está más cerca de cada punto. 
    * *Detalle técnico:* Calculamos los polígonos frame a frame y aplicamos una máscara de recorte (`clip_rect`) para que las regiones de control no se "derramen" fuera de las líneas de banda.
3.  **Capa de Contexto:** Superpone información semántica en tiempo real, como el equipo que tiene la posesión, el resultado final de la jugada (ej: "GOL", "Pérdida") y un cronómetro relativo.

Esta función es el puente final entre el "Big Data" y el ojo del entrenador.

In [ ]:


# Aumentamos límite de memoria para gráficos complejos
matplotlib.rcParams['animation.embed_limit'] = 100.0

def animate_pro_phase(phase_id, phases_df, t_home_df, t_away_df, show_voronoi=False):
    """
    Genera animación de una fase táctica.
    Args:
        phase_id (int): ID de la fase.
        phases_df (pd.DataFrame): Tabla maestra de fases (pro_phases).
        t_home_df (pd.DataFrame): Tracking equipo local.
        t_away_df (pd.DataFrame): Tracking equipo visitante.
        show_voronoi (bool): Activar/Desactivar capa de control espacial.
    """
    
    # 1. VALIDACIÓN DE DATOS
    try:
        phase_row = phases_df[phases_df['Phase_ID'] == phase_id]
        if phase_row.empty:
            print(f"❌ Error: La Fase {phase_id} no existe en el DataFrame proporcionado.")
            return None
        phase = phase_row.iloc[0]
    except Exception as e:
        print(f"❌ Error leyendo la fase: {e}")
        return None

    # Obtener Frames de inicio/fin
    start_f = phase.get('Start_Frame', phase.get('Start Frame'))
    end_f = phase.get('End_Frame', phase.get('End Frame'))
    
    # Margen de contexto (1.5s)
    end_f_ext = int(end_f + 38)
    
    # 2. SLICING (Corte de datos)
    # Usamos los dataframes pasados como argumento
    home_slice = t_home_df[(t_home_df['Frame'] >= start_f) & (t_home_df['Frame'] <= end_f_ext)]
    away_slice = t_away_df[(t_away_df['Frame'] >= start_f) & (t_away_df['Frame'] <= end_f_ext)]
    
    if home_slice.empty:
        print("⚠️ Advertencia: No hay datos de tracking para este rango de frames.")
        return None

    # 3. CONFIGURACIÓN DEL PITCH
    pitch = Pitch(pitch_type='custom', pitch_length=105, pitch_width=68, 
                  pitch_color='#2b2b2b', line_color='white')
    
    fig, ax = pitch.draw(figsize=(10, 6))
    
    # Objetos Gráficos
    scat_h = pitch.scatter([], [], ax=ax, c='#ff4b4b', s=120, edgecolors='white', label='Local', zorder=4)
    scat_a = pitch.scatter([], [], ax=ax, c='#4b88ff', s=120, edgecolors='white', label='Visitante', zorder=4)
    scat_b = pitch.scatter([], [], ax=ax, c='#ffff00', s=90, edgecolors='black', zorder=5)
    
    # Textos
    info_txt = ax.text(52.5, 74, f"{phase['Team']} | {phase['Outcome']}", 
                       ha='center', fontsize=14, fontweight='bold', color='white')
    time_txt = ax.text(105, 74, "0.0s", ha='right', fontsize=12, color='white')

    # Configuración Voronoi
    voronoi_patches = []
    clip_rect = None
    if show_voronoi:
        clip_rect = mpatches.Rectangle((0, 0), 105, 68, transform=ax.transData)

    # 4. FUNCIONES INTERNAS
    def get_xy(row, df):
        xs, ys = [], []
        cols = [c for c in df.columns if c.startswith('Player') and 'Unnamed' not in c]
        for c in cols:
            idx = df.columns.get_loc(c)
            vx = row.iloc[idx]; vy = row.iloc[idx+1]
            if pd.notna(vx): xs.append(vx*105); ys.append(vy*68)
        return xs, ys

    def calc_voronoi(pts, color):
        if len(pts) < 4: return []
        ghosts = [[-20,-20],[-20,90],[130,-20],[130,90]] # Puntos de cierre
        all_pts = np.vstack([pts, ghosts])
        vor = Voronoi(all_pts)
        polys = []
        for i in range(len(pts)):
            region = vor.regions[vor.point_region[i]]
            if -1 in region or not region: continue
            verts = [vor.vertices[v] for v in region]
            polys.append(mpatches.Polygon(verts, facecolor=color, alpha=0.2, edgecolor=color))
        return polys

    # 5. UPDATE LOOP
    def update(i):
        nonlocal voronoi_patches
        if i >= len(home_slice) or i >= len(away_slice): return []
        
        row_h = home_slice.iloc[i]
        row_a = away_slice.iloc[i]
        
        hx, hy = get_xy(row_h, t_home_df)
        ax_x, ax_y = get_xy(row_a, t_away_df)
        
        # Actualizar Posiciones
        scat_h.set_offsets(np.c_[hx, hy])
        scat_a.set_offsets(np.c_[ax_x, ax_y])
        
        # Balón
        bx, by = [], []
        if 'Ball' in t_home_df.columns:
            b_idx = t_home_df.columns.get_loc('Ball')
            val_bx = row_h.iloc[b_idx]; val_by = row_h.iloc[b_idx+1]
            if pd.notna(val_bx): bx, by = [val_bx*105], [val_by*68]
        if bx: scat_b.set_offsets(np.c_[bx, by])
        else: scat_b.set_offsets(np.zeros((0, 2)))
        
        # Tiempo
        start_t = phase.get('Start_Time', phase.get('Start Time [s]'))
        time_txt.set_text(f"{row_h['Time [s]'] - start_t:.1f}s")
        
        artists = [scat_h, scat_a, scat_b, info_txt, time_txt]
        
        # Lógica Voronoi
        if show_voronoi:
            for p in voronoi_patches: p.remove()
            voronoi_patches = []
            
            if hx and ax_x:
                pts_h = np.column_stack((hx, hy))
                pts_a = np.column_stack((ax_x, ax_y))
                
                polys = calc_voronoi(pts_h, '#ff4b4b') + calc_voronoi(pts_a, '#4b88ff')
                for p in polys:
                    ax.add_patch(p)
                    p.set_clip_path(clip_rect)
                    voronoi_patches.append(p)
            artists.extend(voronoi_patches)
            
        return artists

    anim = FuncAnimation(fig, update, frames=len(home_slice), interval=40, blit=False)
    plt.close()
    return HTML(anim.to_jshtml())

# --- ZONA DE PRUEBAS ---
print("🧪 Probando función corregida...")

# Aseguramos que pro_phases existe. Si no, avisa al usuario.
if 'pro_phases' in locals():
    # Cogemos la primera fase disponible
    test_id = pro_phases.iloc[0]['Phase_ID']
    print(f"🎬 Generando vídeo (ID {test_id}) | Voronoi: ACTIVADO")
    
    # ¡AQUÍ ESTÁ LA CLAVE! Pasamos las tablas explícitamente
    video = animate_pro_phase(
        phase_id=test_id, 
        phases_df=pro_phases,       # <-- Pasamos la tabla de fases
        t_home_df=tracking_home,    # <-- Pasamos tracking local
        t_away_df=tracking_away,    # <-- Pasamos tracking visitante
        show_voronoi=True           # <-- Activamos Voronoi
    )
    display(video)
else:
    print("❌ ERROR: La variable 'pro_phases' no existe.")
    print("Por favor, ejecuta primero la Celda 2 (Motor de Datos) para crear las fases.")

In [ ]:
video = animate_pro_phase(
        phase_id=5, 
        phases_df=pro_phases,       # <-- Pasamos la tabla de fases
        t_home_df=tracking_home,    # <-- Pasamos tracking local
        t_away_df=tracking_away,    # <-- Pasamos tracking visitante
        show_voronoi=True           # <-- Activamos Voronoi
    )

display(video)

In [ ]:
video = animate_pro_phase(
        phase_id=5, 
        phases_df=pro_phases,       # <-- Pasamos la tabla de fases
        t_home_df=tracking_home,    # <-- Pasamos tracking local
        t_away_df=tracking_away,    # <-- Pasamos tracking visitante
        show_voronoi=False           # <-- Activamos Voronoi
    )

display(video)

# 5. Cálculo de KPIs Tácticos Avanzados (Master Engine)

Esta celda es el **corazón analítico** del proyecto. Aquí transformamos las coordenadas crudas en métricas de rendimiento de alto nivel, integrando geometría y modelos de valor.

**Dimensiones calculadas:**
1.  **Valor del Juego (EPV):** Utilizamos una rejilla de *Expected Possession Value* para medir cuánto aumenta la probabilidad de gol según dónde se mueve el balón. 
2.  **Zonificación (Juego de Posición):** Mapeamos cada acción a las 18 zonas estándar (3 carriles verticales $\times$ 6 alturas) para identificar patrones espaciales. 
3.  **Estructura Defensiva:**
    * **Altura de Bloque:** Distancia media de la defensa a su propia portería.
    * **Compacidad:** Calculamos el *Convex Hull* (polígono envolvente) de los jugadores para medir cuánto espacio ocupan. 
4.  **Packing:** Contamos cuántos rivales son "eliminados" (superados por la línea del balón) en cada fase ofensiva.

El resultado es el `final_kpi_dataset`, una tabla maestra donde cada fila es una jugada con todas sus características físicas y tácticas calculadas.

In [ ]:


# ---------------------------------------------------------
# 1. PREPARACIÓN DE DATOS DE VALOR (EPV GRID)
# ---------------------------------------------------------
# Intentamos cargar el grid de EPV. Si no existe, creamos uno dummy para que el código no falle.
epv_filename = './epv_data.csv'
if not os.path.exists(epv_filename):
    print("⚠️ Aviso: No se encontró 'epv_data.csv'. Creando uno por defecto...")
    # Grid genérico simplificado 32x50
    dummy_grid = np.linspace(0.005, 0.05, 50) * np.ones((32, 50))
    pd.DataFrame(dummy_grid).to_csv(epv_filename, header=False, index=False)

epv_grid = pd.read_csv(epv_filename, header=None).values

# ---------------------------------------------------------
# 2. FUNCIONES AUXILIARES (ZONAS Y EPV)
# ---------------------------------------------------------
def get_zone_metrica(x, y):
    """Devuelve la Zona (1-18), Tercio y Carril según coordenadas 0-1"""
    # Grid 3x6 (18 Zonas)
    col = int(x // (1.0 / 6)); col = min(max(col, 0), 5)
    row = int(y // (1.0 / 3)); row = min(max(row, 0), 2)
    
    # Matriz de Zonas (Standard)
    labels = [[3, 6, 9, 12, 15, 18], [2, 5, 8, 11, 14, 17], [1, 4, 7, 10, 13, 16]]
    zone_num = labels[row][col]
    
    # Semántica de Tercios
    if x < 1/3: tercio = 'Defensivo'
    elif x < 2/3: tercio = 'Medio'
    else: tercio = 'Ofensivo'
    
    # Semántica de Carriles (Y=0 Top, Y=1 Bottom en Metrica Standard)
    if y < 0.21: inter = 'Banda Izq'
    elif y < 0.37: inter = 'Pasillo Izq'
    elif y < 0.63: inter = 'Central'
    elif y < 0.79: inter = 'Pasillo Der'
    else: inter = 'Banda Der'
    
    return zone_num, tercio, inter

def get_epv(x, y, grid):
    """Obtiene el valor EPV de una coordenada específica"""
    rows, cols = grid.shape
    c = int(x * cols); c = min(max(c, 0), cols-1)
    r = int(y * rows); r = min(max(r, 0), rows-1)
    return grid[r, c]

# ---------------------------------------------------------
# 3. MOTOR DE CÁLCULO DE KPIs (Función Maestra)
# ---------------------------------------------------------
def generate_full_tactical_report(phases_df, tracking_home, tracking_away, epv_grid):
    full_report = []
    
    # Identificar columnas de jugadores (excluyendo Ball y Time)
    h_cols = [c for c in tracking_home.columns if c.startswith('Player') and 'Unnamed' not in c]
    a_cols = [c for c in tracking_away.columns if c.startswith('Player') and 'Unnamed' not in c]
    
    print(f"⚙️ Calculando KPIs Tácticos para {len(phases_df)} fases...")
    
    for idx, row in phases_df.iterrows():
        try:
            team = row['Team']
            period = row['Period']
            
            # A. SLICE DE TRACKING (Cortar la "película" de la jugada)
            t_home = tracking_home[(tracking_home['Frame'] >= row['Start_Frame']) & (tracking_home['Frame'] <= row['End_Frame'])]
            t_away = tracking_away[(tracking_away['Frame'] >= row['Start_Frame']) & (tracking_away['Frame'] <= row['End_Frame'])]
            
            if t_home.empty: continue
            
            # B. NORMALIZACIÓN DE DIRECCIÓN
            # Queremos que X=0 sea SIEMPRE "Mi Portería" y X=1 "Portería Rival"
            # Metrica Standard: P1 Home->1, Away->0. P2 Swap.
            if period == 1:
                home_attacks_1 = True # Home ataca hacia 1
                goal_x = 1.0 if team == 'Home' else 0.0
            else:
                home_attacks_1 = False # Home ataca hacia 0
                goal_x = 0.0 if team == 'Home' else 1.0
            
            # C. DATOS DEL BALÓN
            if 'Ball' in t_home.columns:
                b_idx = t_home.columns.get_loc('Ball')
                bx_raw = t_home.iloc[:, b_idx].values
                by_raw = t_home.iloc[:, b_idx+1].values
                
                # Normalizar Coordenadas del Balón (Para cálculos ofensivos)
                # Si el equipo ataca hacia 1, dejamos X como está. Si ataca hacia 0, invertimos (1-X).
                if (team == 'Home' and home_attacks_1) or (team == 'Away' and not home_attacks_1):
                    norm_bx, norm_by = bx_raw, by_raw
                else:
                    norm_bx, norm_by = 1.0 - bx_raw, 1.0 - by_raw
            else:
                continue # Sin balón no hay métricas válidas
            
            # Filtrar NaNs (Balón fuera de campo o no trackeado)
            valid_idx = np.where(~np.isnan(norm_bx))[0]
            if len(valid_idx) < 2: continue
            
            # Puntos de Inicio (Start) y Fin (End) limpios
            sx, sy = norm_bx[valid_idx[0]], norm_by[valid_idx[0]]
            ex, ey = norm_bx[valid_idx[-1]], norm_by[valid_idx[-1]]
            
            # --- DIMENSIÓN 1: DINÁMICA (Velocidad y Verticalidad) ---
            dist_gained_m = (ex - sx) * 105 # Metros ganados hacia portería
            speed_ms = dist_gained_m / row['Duration'] if row['Duration'] > 0 else 0
            
            # Directness (Distancia Neta / Distancia Total Recorrida)
            # Calculamos la distancia euclidiana paso a paso
            step_dists = np.sqrt(np.diff(norm_bx*105)**2 + np.diff(norm_by*68)**2)
            total_dist = np.nansum(step_dists)
            net_dist = np.sqrt(((ex-sx)*105)**2 + ((ey-sy)*68)**2)
            directness = net_dist / total_dist if total_dist > 0 else 0
            
            tempo = (row['Event_Count'] / row['Duration']) * 60 # Eventos por minuto
            
            # --- DIMENSIÓN 2: ESTRUCTURA RIVAL (Bloque y Compacidad) ---
            # Identificar quién defiende
            rival_tracking = t_away if team == 'Home' else t_home
            rival_cols = a_cols if team == 'Home' else h_cols
            
            # Extraer todas las X de los rivales en una matriz (Players, Frames)
            rival_xs = []
            rival_ys = [] # Necesario para ConvexHull
            for c in rival_cols:
                idx = rival_tracking.columns.get_loc(c)
                rival_xs.append(rival_tracking.iloc[:, idx].values)
                rival_ys.append(rival_tracking.iloc[:, idx+1].values)
            rival_xs = np.array(rival_xs)
            rival_ys = np.array(rival_ys)
            
            # Altura del Bloque (Block Height): Distancia a SU línea de gol
            # Si goal_x=1 (Yo ataco 1), Rival defiende 1. Distancia = 1 - X_rival.
            if goal_x == 1.0: 
                dists_to_goal = np.abs(1.0 - rival_xs)
            else:
                dists_to_goal = np.abs(rival_xs)
            
            # Promedio de la línea defensiva (Top 3 jugadores más atrasados)
            dists_to_goal = np.nan_to_num(dists_to_goal, nan=999)
            sorted_dists = np.sort(dists_to_goal, axis=0) # Ordenamos por cercanía a gol
            avg_block_m = np.nanmean(sorted_dists[1:4, :]) * 105 # Media frame a frame
            
            # Compacidad (Convex Hull en el frame central de la jugada)
            mid_idx = rival_xs.shape[1] // 2
            mid_xs = rival_xs[:, mid_idx]
            mid_ys = rival_ys[:, mid_idx]
            points = np.column_stack((mid_xs, mid_ys))
            # Limpieza de puntos inválidos
            points = points[~np.isnan(points).any(axis=1)] 
            points = points[points[:,0] != 999] 
            
            compactness_m2 = 0
            if len(points) > 2:
                try: compactness_m2 = ConvexHull(points).area * 105 * 68
                except: pass
            
            # --- DIMENSIÓN 3: ESPACIO (Zonas y Carriles) ---
            # Carril dominante (Media de Y del balón)
            mean_y = np.nanmean(norm_by * 68)
            if mean_y < 21: lane = "Izquierda"
            elif mean_y > 47: lane = "Derecha"
            else: lane = "Centro"
            
            # Zonas Metrica (Start y End)
            s_zone_n, s_tercio, s_inter = get_zone_metrica(sx, sy)
            e_zone_n, e_tercio, e_inter = get_zone_metrica(ex, ey)
            
            # Path Zones (Por dónde pasó el balón - Muestreo)
            path_zones = set()
            for bx, by in zip(norm_bx[::25], norm_by[::25]): # Muestreo cada ~1s
                if pd.notna(bx):
                    z, _, _ = get_zone_metrica(bx, by)
                    path_zones.add(z)
            
            # --- DIMENSIÓN 4: VALOR (EPV y Packing) ---
            # EPV Added (Valor final - Valor inicial)
            epv_s = get_epv(sx, sy, epv_grid)
            epv_e = get_epv(ex, ey, epv_grid)
            epv_val = epv_e - epv_s
            
            # Packing (Rivales superados)
            # Usamos coordenadas normalizadas de rivales para comparar con balón normalizado
            if (team == 'Home' and home_attacks_1) or (team == 'Away' and not home_attacks_1):
                rival_norm_xs = rival_xs
            else:
                rival_norm_xs = 1.0 - rival_xs
            
            # Contamos rivales que están "Detrás del balón" (X_Rival < X_Balon)
            packed_start = np.sum(rival_norm_xs[:, 0] < sx)
            packed_end = np.sum(rival_norm_xs[:, -1] < ex)
            packing_val = packed_end - packed_start
            
            # --- GUARDAR TODO EN EL DICCIONARIO ---
            full_report.append({
                'Phase_ID': row['Phase_ID'],
                'Team': team,
                'Duration': round(row['Duration'], 1),
                'Speed_ms': round(speed_ms, 2),
                'Directness': round(directness, 2),
                'Tempo': round(tempo, 1),
                'Block_Height_m': round(avg_block_m, 1),
                'Compactness_m2': int(compactness_m2),
                'Lane': lane,
                'Start_Zone': s_zone_n,
                'Start_Desc': f"{s_tercio} ({s_inter})",
                'End_Zone': e_zone_n,
                'End_Desc': f"{e_tercio} ({e_inter})",
                'Path_Zones': sorted(list(path_zones)),
                'Packing': int(packing_val),
                'EPV_Added': round(epv_val, 4),
                'Outcome': row['Outcome']
            })
            
        except Exception:
            continue # Saltamos errores puntuales para no detener el proceso
            
    return pd.DataFrame(full_report)

# ---------------------------------------------------------
# 4. EJECUCIÓN
# ---------------------------------------------------------
# Asegúrate de tener 'pro_phases' creado (Celda anterior)
final_kpi_dataset = generate_full_tactical_report(pro_phases, tracking_home, tracking_away, epv_grid)

print(f"✅ Dataset Maestro Creado: {len(final_kpi_dataset)} jugadas procesadas.")
display(final_kpi_dataset.head(5))

# Motor de IA (Vectorización y Búsqueda)

## Generador de Narrativa Natural (El "Narrador")

Esta celda contiene la lógica de **NLG (Natural Language Generation)**. Su función es traducir los KPIs numéricos fríos en una crónica textual que parezca escrita por un analista humano.

**¿Por qué es vital?**
Para que la Inteligencia Artificial (en pasos posteriores) pueda "entender" el fútbol, necesita texto, no solo números. Esta función crea esa capa semántica.

**Reglas de Traducción:**
1.  **Dinámica:** Si la velocidad supera los 5.5 m/s, lo etiqueta como *"Contraataque explosivo"*; si es baja, *"Posesión de control"*.
2.  **Estructura:** Interpreta la altura del bloque defensivo rival (ej: < 25m es *"Bloque bajo hundido"*).
3.  **Impacto:** Verbaliza métricas complejas como el EPV o Packing (ej: *"Rompiendo 4 líneas de presión"*).

El resultado se guarda en la columna `Tactical_Text`, que será la fuente de conocimiento para nuestro buscador semántico.

In [ ]:

def generate_tactical_summary(row):
    # ---------------------------------------------------------
    # 1. ZONAS Y DESCRIPCIÓN GEOGRÁFICA
    # ---------------------------------------------------------
    # Usamos las columnas que generamos en el motor de KPIs
    start_txt = f"Zona {row['Start_Zone']} ({row['Start_Desc']})"
    end_txt = f"Zona {row['End_Zone']} ({row['End_Desc']})"

    # ---------------------------------------------------------
    # 2. INTERPRETACIÓN DE DINÁMICA (Velocidad)
    # ---------------------------------------------------------
    if row['Speed_ms'] > 5.5:
        tipo_ataque = "Contraataque explosivo"
    elif row['Speed_ms'] > 2.5:
        tipo_ataque = "Ataque progresivo rápido"
    else:
        tipo_ataque = "Posesión de control"
        
    # Verticalidad
    if row['Directness'] > 0.85: estilo = "muy vertical"
    elif row['Directness'] < 0.6: estilo = "horizontal"
    else: estilo = "equilibrado"

    # ---------------------------------------------------------
    # 3. ESTRUCTURA RIVAL
    # ---------------------------------------------------------
    if row['Block_Height_m'] < 25:
        bloque = "bloque bajo hundido"
    elif row['Block_Height_m'] > 40:
        bloque = "bloque alto / presión"
    else:
        bloque = "bloque medio"
        
    compacidad = "compacto" if row['Compactness_m2'] < 2500 else "disperso"

    # ---------------------------------------------------------
    # 4. VALOR (Packing y EPV)
    # ---------------------------------------------------------
    impacto = []
    if row['Packing'] >= 4: impacto.append(f"rompiendo {row['Packing']} líneas")
    
    if row['EPV_Added'] > 0.05: impacto.append("generando alto peligro (EPV+)")
    elif row['EPV_Added'] < -0.01: impacto.append("sin profundidad")
    
    impacto_str = " y ".join(impacto) if impacto else "de transición neutra"

    # ---------------------------------------------------------
    # 5. GENERACIÓN FINAL
    # ---------------------------------------------------------
    # Construimos la frase que leerá la IA
    narrativa = (
        f"Jugada del {row['Team']}: {tipo_ataque} {estilo}. "
        f"Inició en {start_txt} y llegó a {end_txt} por carril {row['Lane']}. "
        f"Ante {bloque} {compacidad}. "
        f"Acción {impacto_str}. Resultado: {row['Outcome']}."
    )
    
    return narrativa

# --- EJECUCIÓN ---
print("✍️ Escribiendo narrativas tácticas para cada jugada...", end=" ")

# Aplicamos la función fila por fila
final_kpi_dataset['Tactical_Text'] = final_kpi_dataset.apply(generate_tactical_summary, axis=1)

print("✅ Hecho.")
print("Ejemplo de lo que leerá la IA:")
print(f"👉 {final_kpi_dataset.iloc[0]['Tactical_Text']}")
print(display(final_kpi_dataset.head(5)))


# 7. Motor de Inteligencia Artificial (El Cerebro Semántico)

Esta celda implementa la capa de **NLP (Procesamiento de Lenguaje Natural)**. Aquí es donde el sistema deja de ver simples cadenas de texto y empieza a "entender" el significado de las jugadas.

**Pasos del Proceso:**
1.  **Carga del Modelo:** Utilizamos `SentenceTransformer` (versión multilingüe) para procesar texto en español. Este modelo actúa como un traductor de "Lenguaje Humano" a "Lenguaje Máquina".
2.  **Vectorización (Embeddings):** Convertimos cada narrativa táctica generada en el paso anterior en un vector matemático de alta dimensión.  Esto sitúa jugadas tácticamente similares (ej: "Contraataque" y "Transición rápida") cerca unas de otras en el espacio matemático, aunque usen palabras distintas.
3.  **Búsqueda Semántica:** Definimos la función `buscar_jugadas_ia`, que utiliza la **Similitud del Coseno**.  Esto nos permite encontrar la jugada más parecida a la pregunta del usuario basándonos en el concepto, no en la coincidencia exacta de palabras clave.

In [ ]:


# ==============================================================================
# 1. CONFIGURACIÓN DEL MODELO (SELECTOR)
# ==============================================================================
# Opciones disponibles:
# "fast"   -> 'all-MiniLM-L6-v2' (Muy rápido, optimizado para inglés)
# "pro"    -> 'paraphrase-multilingual-MiniLM-L12-v2' (Mejor para ESPAÑOL)

MODEL_MODE = "pro"  # <--- CAMBIA ESTO A "fast" o "pro" SEGÚN PREFIERAS

if MODEL_MODE == "pro":
    model_name = 'paraphrase-multilingual-MiniLM-L12-v2'
    print(f"🧠 Modo PRO seleccionado: Cargando modelo Multilingüe ({model_name})...")
else:
    model_name = 'all-MiniLM-L6-v2'
    print(f"⚡ Modo FAST seleccionado: Cargando modelo Ligero ({model_name})...")

# Cargar el modelo (La primera vez descargará unos 100-200MB)
start_t = time.time()
model = SentenceTransformer(model_name)
print(f"✅ Modelo cargado en {time.time() - start_t:.2f} segundos.")

# ==============================================================================
# 2. VECTORIZACIÓN (Crear el "Cerebro" de la base de datos)
# ==============================================================================
# Verificamos que existen los datos
if 'final_kpi_dataset' not in locals() or 'Tactical_Text' not in final_kpi_dataset.columns:
    print("❌ ERROR: No se encuentra 'final_kpi_dataset' o la columna 'Tactical_Text'.")
    print("Ejecuta primero la celda del 'Narrador Táctico'.")
else:
    print("🧮 Vectorizando narrativas tácticas...", end=" ")
    
    # Extraemos textos
    textos = final_kpi_dataset['Tactical_Text'].tolist()
    
    # Encode crea los embeddings (Vectores numéricos)
    embeddings_jugadas = model.encode(textos, show_progress_bar=True)
    
    print("✅ Base de Datos Vectorial Lista.")
    print(f"   Dimensiones: {embeddings_jugadas.shape} (Jugadas x Neuronas)")

# ==============================================================================
# 3. FUNCIÓN DE BÚSQUEDA SEMÁNTICA
# ==============================================================================
def buscar_jugadas_ia(query, top_k=5):
    """
    Recibe una frase natural, la vectoriza y busca las jugadas más similares
    en el espacio vectorial mediante similitud del coseno.
    """
    # 1. Vectorizar la pregunta del usuario con el MISMO modelo
    query_vec = model.encode([query])
    
    # 2. Calcular distancia (similitud) con todas las jugadas almacenadas
    similitudes = cosine_similarity(query_vec, embeddings_jugadas)[0]
    
    # 3. Obtener los índices de los mejores resultados (de mayor a menor score)
    top_indices = np.argsort(similitudes)[-top_k:][::-1]
    
    # 4. Construir respuesta
    resultados = []
    for idx in top_indices:
        score = similitudes[idx]
        row = final_kpi_dataset.iloc[idx]
        resultados.append({
            'Phase_ID': row['Phase_ID'],
            'Score': round(score, 4), # 0 a 1
            'Team': row['Team'],
            'Resultado': row['Outcome'],
            'Narrativa_IA': row['Tactical_Text']
        })
        
    return pd.DataFrame(resultados)

# ==============================================================================
# 4. PRUEBA INMEDIATA
# ==============================================================================
print("\n🔎 PRUEBA DE BÚSQUEDA AUTOMÁTICA:")
prompt_prueba = "Contraataque rápido por banda acabando en gol"
print(f"Prompt: '{prompt_prueba}'\n")

# Ejecutar búsqueda
df_res = buscar_jugadas_ia(prompt_prueba)

# Mostrar resultados limpios
display(df_res[['Phase_ID', 'Team', 'Score', 'Narrativa_IA']])


# Interfaz Final: Asistente Táctico con IA (Dashboard)

Esta celda consolida todo el trabajo anterior en un **Producto Mínimo Viable (MVP)**. Utilizamos la librería `ipywidgets` para crear una interfaz gráfica que oculta la complejidad del código y ofrece una experiencia de usuario fluida.

**Flujo de Interacción:**
1.  **Prompt Táctico:** El usuario describe lo que busca en lenguaje natural (ej: *"Contraataque rápido rompiendo líneas"*).
2.  **Búsqueda Semántica:** El sistema vectoriza la consulta y recupera las 3 jugadas más similares de la base de datos.
3.  **Renderizado On-Demand:** Al seleccionar una jugada, el sistema:
    * Ejecuta `animate_pro_phase` para generar el video con capas de **Voronoi** e **Inercia**.
    * Recupera la narrativa del `Tactical_Text` y la formatea en un **Informe HTML** estilizado con las métricas clave (Velocidad, EPV, Zonas).

Es la demostración final de cómo la IA Generativa y la Analítica de Datos pueden transformar el scouting deportivo.

In [ ]:


# ---------------------------------------------------------
# 1. CONFIGURACIÓN DE LA INTERFAZ
# ---------------------------------------------------------
print("🚀 Iniciando Asistente Táctico...")

# Widgets de Entrada
txt_search = widgets.Text(
    value='',
    placeholder='Ej: Contraataque rápido rompiendo líneas...',
    description='🔍 Prompt:',
    layout={'width': '60%'}
)

btn_search = widgets.Button(
    description=' Consultar IA',
    button_style='success', # Color verde 'success'
    icon='robot',
    layout={'width': '150px'}
)

# Zonas de Salida
output_list = widgets.Output()   # Lista de resultados textuales
output_visual = widgets.Output() # Video e Informe detallado

# ---------------------------------------------------------
# 2. LÓGICA DEL BUSCADOR
# ---------------------------------------------------------
def on_search_click(b):
    query = txt_search.value
    if not query: return
    
    with output_list:
        clear_output()
        print(f"🧠 La IA está analizando la petición: '{query}'...")
        
        # 1. Llamamos a TU función de vectorización (definida en el paso anterior)
        try:
            df_res = buscar_jugadas_ia(query, top_k=3)
        except NameError:
            print("❌ Error: No se encuentra 'buscar_jugadas_ia'. Ejecuta la celda de Vectorización primero.")
            return

        if df_res.empty:
            print("⚠️ No se encontraron coincidencias semánticas.")
            return
            
        print(f"✅ Se encontraron {len(df_res)} jugadas similares.\n")
        
        # 2. Generar Botones para cada resultado
        for _, row in df_res.iterrows():
            score_pct = row['Score'] * 100
            desc_btn = f"ID {row['Phase_ID']} | {row['Team']} | {row['Resultado']} (Coincidencia: {score_pct:.1f}%)"
            
            btn = widgets.Button(
                description=desc_btn, 
                layout={'width': '98%'}, 
                button_style='info', # Azul
                icon='play-circle'
            )
            
            # --- LÓGICA AL ELEGIR UN RESULTADO (CLOSURE) ---
            def on_res_click(btn_obj, pid=row['Phase_ID']):
                with output_visual:
                    clear_output(wait=True)
                    print(f"🎬 Renderizando vídeo táctico para Fase {pid}...")
                    
                    try:
                        # A. GENERAR VIDEO (Usamos la función robusta con Voronoi activado)
                        # Nota: Pasamos tracking_home y tracking_away explícitamente
                        video = animate_pro_phase(
                            phase_id=pid,
                            phases_df=pro_phases,
                            t_home_df=tracking_home,
                            t_away_df=tracking_away,
                            show_voronoi=True # ¡Activamos el modo Elite!
                        )
                        if video: display(video)
                        
                        # B. GENERAR INFORME HTML
                        # Recuperamos los datos de esa fila específica
                        data_row = final_kpi_dataset[final_kpi_dataset['Phase_ID'] == pid].iloc[0]
                        text_ia = data_row['Tactical_Text']
                        
                        # Estilos HTML
                        html_report = f"""
                        <div style="font-family: sans-serif; background-color: #ffffff; border: 1px solid #e0e0e0; border-left: 6px solid #28a745; padding: 20px; border-radius: 8px; margin-top: 15px; box-shadow: 0 4px 10px rgba(0,0,0,0.05);">
                            <div style="display: flex; align-items: center; margin-bottom: 15px;">
                                <span style="font-size: 24px; margin-right: 10px;">🤖</span>
                                <h3 style="margin: 0; color: #2c3e50;">Informe de Inteligencia Artificial</h3>
                            </div>
                            
                            <div style="background-color: #f8f9fa; padding: 15px; border-radius: 6px; font-size: 15px; line-height: 1.6; color: #444;">
                                {text_ia.replace('Home', '<b>Local</b>').replace('Away', '<b>Visitante</b>')
                                     .replace('GOL', '<span style="color:#fff; background-color:#d9534f; padding:2px 6px; border-radius:4px; font-weight:bold;">GOL ⚽</span>')
                                     .replace('Contraataque', '<b>Contraataque ⚡</b>')
                                     .replace('rompiendo', '<b>rompiendo 💥</b>')}
                            </div>
                            
                            <hr style="border: 0; border-top: 1px solid #eee; margin: 20px 0;">
                            
                            <div style="display: flex; justify-content: space-between; text-align: center; color: #555;">
                                <div style="flex: 1;">
                                    <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; color: #888;">Inicio</div>
                                    <div style="font-weight: bold; font-size: 14px;">Zona {data_row['Start_Zone']}</div>
                                </div>
                                <div style="flex: 1; border-left: 1px solid #eee; border-right: 1px solid #eee;">
                                    <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; color: #888;">Velocidad</div>
                                    <div style="font-weight: bold; font-size: 14px;">{data_row['Speed_ms']:.1f} m/s</div>
                                </div>
                                <div style="flex: 1;">
                                    <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; color: #888;">Valor (EPV)</div>
                                    <div style="font-weight: bold; font-size: 14px; color: #28a745;">+{data_row['EPV_Added']:.3f}</div>
                                </div>
                            </div>
                        </div>
                        """
                        display(HTML(html_report))
                        
                    except Exception as e:
                        print(f"❌ Error visualizando: {e}")
                        import traceback
                        traceback.print_exc()

            btn.on_click(on_res_click)
            display(btn)
            
            # Pequeña previsualización del texto
            print(f"   ↳ {row['Narrativa_IA'][:90]}...") 
            print("")

# Conectar el botón
btn_search.on_click(on_search_click)

# ---------------------------------------------------------
# 3. VISUALIZACIÓN
# ---------------------------------------------------------
header = widgets.HTML("<h2>⚽ Tactical AI Assistant <span style='font-size:14px; color:gray;'>Semantic Search Engine</span></h2>")
display(header)
display(widgets.HBox([txt_search, btn_search]))
display(widgets.HTML("<hr style='margin-top: 20px; margin-bottom: 20px;'>"))

# Grid layout: Resultados a la izquierda, Video a la derecha (si hay espacio)
# Para notebooks simples, mejor uno debajo del otro.
display(output_list)
display(output_visual)

# Visualización Macro: ADN del Partido (Match DNA)

Antes de analizar jugadas individuales, necesitamos entender la narrativa global del encuentro. Esta función genera un **Tablero de Control Estratégico** con tres paneles gráficos:

1.  **Mapa de Estilo (Scatter Plot):** Cruza *Velocidad* vs. *Verticalidad*. Nos permite identificar rápidamente la identidad del equipo:
    * *Arriba a la derecha:* Contraataque directo.
    * *Abajo a la izquierda:* Juego de posesión y control horizontal.
2.  **Altura del Bloque (Bar Chart):** Visualiza la agresividad defensiva. ¿El equipo presiona en campo rival (Bloque Alto > 40m) o espera atrás (Bloque Bajo < 25m)?
3.  **Ritmo de Peligro (Line Chart):** Muestra la suma acumulada de **EPV** (Expected Possession Value) a lo largo del tiempo. Es vital para detectar "momentums" de dominio: si la curva sube rápido, el equipo estaba asediando al rival.

In [ ]:


def plot_match_dna(kpi_df):
    if kpi_df.empty: return

    # 1. Agrupar datos por equipo
    match_stats = kpi_df.groupby('Team').agg({
        'Speed_ms': 'mean',
        'Directness': 'mean',
        'Block_Height_m': 'mean',
        'Packing': 'sum',
        'EPV_Added': 'sum',
        'Phase_ID': 'count'
    }).rename(columns={'Phase_ID': 'Total Possessions'})

    # 2. Visualización
    plt.style.use('dark_background')
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    colors = {'Home': '#ff4b4b', 'Away': '#4b88ff'}
    
    # A) Mapa de Estilo (Scatter)
    sns.scatterplot(data=kpi_df, x='Speed_ms', y='Directness', hue='Team', 
                    palette=colors, ax=axes[0], alpha=0.5, s=80)
    axes[0].set_title('🧬 ESTILO: Velocidad vs Verticalidad', fontsize=12, color='white')
    axes[0].set_xlabel('Velocidad (m/s)')
    axes[0].set_ylabel('Verticalidad (0-1)')
    axes[0].legend(loc='upper right')

    # B) Altura del Bloque (Bar)
    match_stats['Block_Height_m'].plot(kind='bar', color=[colors.get(x, 'grey') for x in match_stats.index], ax=axes[1], alpha=0.8)
    axes[1].set_title('🛡️ ALTURA BLOQUE DEFENSIVO', fontsize=12, color='white')
    axes[1].set_ylabel('Metros desde portería propia')
    axes[1].tick_params(axis='x', rotation=0)

    # C) Ritmo de Peligro (Line - Cumulative EPV)
    for team in match_stats.index:
        team_data = kpi_df[kpi_df['Team'] == team]
        # Eje X artificial (número de jugada)
        x_axis = range(len(team_data))
        y_axis = np.cumsum(team_data['EPV_Added'].values)
        axes[2].plot(x_axis, y_axis, label=team, color=colors.get(team, 'white'), linewidth=3)
    
    axes[2].set_title('📈 RITMO DE GENERACIÓN DE PELIGRO (EPV)', fontsize=12, color='white')
    axes[2].set_xlabel('Secuencia de Jugadas')
    axes[2].set_ylabel('EPV Acumulado')
    axes[2].legend()
    axes[2].grid(alpha=0.2)

    plt.tight_layout()
    plt.show()
    
    print("\n📊 TABLA RESUMEN:")
    display(match_stats.style.background_gradient(cmap='Greens'))

# EJECUTAR
plot_match_dna(final_kpi_dataset)

# Motor de Física: Cálculo de Inercia y Velocidad (Pre-Cálculo de Vectores)

Los datos originales de tracking son "estáticos": solo nos dicen dónde está un jugador en cada instante, pero no hacia dónde va ni con qué intención.

Esta función transforma las coordenadas posicionales en **Vectores de Movimiento** ($V_x, V_y$):
1.  **Física Básica:** Aplica la derivada de la posición respecto al tiempo ($v = \Delta d / \Delta t$).
2.  **Conversión a Metros:** Transforma las coordenadas normalizadas (0-1) a dimensiones reales del campo (105x68m) para obtener velocidades en metros/segundo.
3.  **Reducción de Ruido:** Aplica una técnica de **suavizado** (`rolling mean`) para eliminar las vibraciones típicas de los sistemas de tracking, logrando que las flechas de inercia sean estables y fluidas en la visualización.

In [ ]:
def calculate_velocities(tracking_df, fps=25, smooth_window=7):
    df = tracking_df.copy()
    dt = 1 / fps
    player_cols = [c for c in df.columns if c.startswith('Player') and 'Unnamed' not in c]
    
    print(f"   ⚙️ Procesando física para {len(player_cols)} jugadores...", end=" ")
    
    for col in player_cols:
        try:
            col_idx = df.columns.get_loc(col)
            if col_idx + 1 < len(df.columns):
                # Calcular velocidad cruda
                dx = df.iloc[:, col_idx].diff() * 105 
                dy = df.iloc[:, col_idx+1].diff() * 68 
                
                # Calcular y Suavizar
                vx = (dx / dt).rolling(window=smooth_window).mean().fillna(0)
                vy = (dy / dt).rolling(window=smooth_window).mean().fillna(0)
                
                # Guardar en el dataframe
                df[f'{col}_vx'] = vx
                df[f'{col}_vy'] = vy
        except: continue
    
    print("Hecho.")
    return df

# EJECUTAR CÁLCULOS
print("🚀 Iniciando Motor de Física...")
tracking_home_phys = calculate_velocities(tracking_home)
tracking_away_phys = calculate_velocities(tracking_away)


# Visualización de Inercia y Física (Pitch Control, con modos 'All' y 'Smart')

Esta celda introduce una capa de visualización basada en vectores físicos. A diferencia de la animación estándar que muestra *dónde* están los jugadores, esta muestra *hacia dónde van y con qué intensidad*.

**Características Clave:**
1.  **Vectores de Velocidad (Flechas):** Utilizamos gráficos tipo `quiver` para dibujar flechas que representan la dirección y magnitud de la carrera de cada jugador. Una flecha larga indica un sprint; una corta, un trote.
2.  **Filtrado Inteligente (Modo Smart):** Para evitar el ruido visual de 22 flechas en pantalla, implementamos un filtro que solo muestra los vectores si la velocidad supera los 2 m/s. Esto resalta tácticamente a los jugadores que están atacando espacios o recuperando posición activamente.
3.  **Estela del Balón:** Añadimos un "rastro" visual a la pelota (últimos 15 frames) para facilitar el seguimiento de la trayectoria de los pases y tiros en la animación.

In [ ]:

def animate_inertial_safe(phase_id, display_mode='smart'):
    """
    display_mode: 
      'all'   -> Muestra flechas de todos los jugadores.
      'smart' -> Solo muestra flechas si el jugador se mueve rápido (> 2 m/s).
                 Limpia mucho la imagen.
    """
    
    # 1. Obtener Datos
    try:
        phase = pro_phases[pro_phases['Phase_ID'] == phase_id].iloc[0]
        start_f = phase['Start_Frame']
        end_f = phase['End_Frame'] + 30 
    except: return None

    # Slice usando los datos FÍSICOS (_phys)
    t_home = tracking_home_phys[(tracking_home_phys['Frame'] >= start_f) & (tracking_home_phys['Frame'] <= end_f)]
    t_away = tracking_away_phys[(tracking_away_phys['Frame'] >= start_f) & (tracking_away_phys['Frame'] <= end_f)]
    
    if t_home.empty: return None

    # 2. Setup Pitch
    pitch = Pitch(pitch_type='custom', pitch_length=105, pitch_width=68, 
                  pitch_color='#2b2b2b', line_color='white')
    fig, ax = pitch.draw(figsize=(10, 6))
    
    # Objetos
    quiver_arts = []
    # Usamos zorder alto para que se vea encima del césped
    scat_h = pitch.scatter([], [], ax=ax, c='#ff4b4b', s=100, edgecolors='w', zorder=4)
    scat_a = pitch.scatter([], [], ax=ax, c='#4b88ff', s=100, edgecolors='w', zorder=4)
    scat_b = pitch.scatter([], [], ax=ax, c='#ffff00', s=80, edgecolors='k', zorder=5)
    
    # Estela del balón (Trail)
    ball_trail, = ax.plot([], [], color='#ffff00', alpha=0.5, linewidth=2, zorder=3)
    ball_history_x, ball_history_y = [], []
    
    info = ax.text(52.5, 74, f"Inercia ({display_mode}) | {phase['Team']}", ha='center', color='white', fontsize=14, fontweight='bold')

    # Helper para extraer posición Y velocidad
    def get_pv(row, df):
        xs, ys, vxs, vys = [], [], [], []
        cols = [c for c in df.columns if c.startswith('Player') and '_' not in c and 'Unnamed' not in c]
        
        for c in cols:
            idx = df.columns.get_loc(c)
            # Extraer pos y vel
            x = row.iloc[idx]
            y = row.iloc[idx+1]
            vx = row.get(f'{c}_vx', 0)
            vy = row.get(f'{c}_vy', 0)
            
            if pd.notna(x):
                xs.append(x * 105)
                ys.append(y * 68)
                vxs.append(vx)
                vys.append(vy)
        return xs, ys, vxs, vys

    def update(i):
        # Limpiar flechas viejas
        for q in quiver_arts: q.remove()
        quiver_arts.clear()
        
        if i >= len(t_home): return []
        
        h_row = t_home.iloc[i]; a_row = t_away.iloc[i]
        
        # Obtener datos completos
        hx, hy, hvx, hvy = get_pv(h_row, tracking_home_phys)
        ax_x, ax_y, avx, avy = get_pv(a_row, tracking_away_phys)
        
        # Actualizar Puntos
        scat_h.set_offsets(np.c_[hx, hy])
        scat_a.set_offsets(np.c_[ax_x, ax_y])
        
        # --- LÓGICA DE FLECHAS INTELIGENTES ---
        def draw_quivers(xs, ys, vxs, vys, color):
            valid_x, valid_y, valid_u, valid_v = [], [], [], []
            for j in range(len(xs)):
                # Calcular magnitud de velocidad (Pitágoras)
                speed = (vxs[j]**2 + vys[j]**2)**0.5
                
                should_draw = False
                if display_mode == 'all':
                    should_draw = True
                elif display_mode == 'smart' and speed > 2.0: # Umbral de 2 m/s
                    should_draw = True
                
                if should_draw:
                    valid_x.append(xs[j])
                    valid_y.append(ys[j])
                    valid_u.append(vxs[j])
                    valid_v.append(vys[j])
            
            if valid_x:
                # Dibujamos todas las flechas de este equipo de una vez
                q = ax.quiver(valid_x, valid_y, valid_u, valid_v, color=color, 
                              scale=20, width=0.004, headwidth=3, alpha=0.8, zorder=3)
                quiver_arts.append(q)

        draw_quivers(hx, hy, hvx, hvy, '#ff4b4b') # Home
        draw_quivers(ax_x, ax_y, avx, avy, '#4b88ff') # Away
        
        # Balón con Estela
        if 'Ball' in tracking_home_phys.columns:
            b_idx = tracking_home_phys.columns.get_loc('Ball')
            bx, by = h_row.iloc[b_idx]*105, h_row.iloc[b_idx+1]*68
            if pd.notna(bx): 
                scat_b.set_offsets(np.c_[bx, by])
                # Guardar historia para la estela (últimos 15 frames)
                ball_history_x.append(bx)
                ball_history_y.append(by)
                if len(ball_history_x) > 15:
                    ball_history_x.pop(0)
                    ball_history_y.pop(0)
                ball_trail.set_data(ball_history_x, ball_history_y)
            
        return [scat_h, scat_a, scat_b, ball_trail, info] + quiver_arts

    anim = FuncAnimation(fig, update, frames=range(0, len(t_home), 2), interval=60, blit=False)
    plt.close()
    return HTML(anim.to_jshtml())

# --- PRUEBA ---
print("🧪 Probando Modo 'Smart' (Solo jugadores rápidos)...")
try:
    if 'pro_phases' in locals():
        test_id = pro_phases.iloc[5]['Phase_ID'] # Probamos una jugada intermedia
        display(animate_inertial_safe(test_id, display_mode='smart'))
    else:
        print("⚠️ Carga 'pro_phases' primero.")
except Exception as e:
    print(f"Error: {e}")

# Análisis de Estructura: Red de Pases (Pass Network)

Esta celda genera una visualización de grafos que revela las conexiones tácticas entre los jugadores. A diferencia del análisis de jugadas aisladas, esto muestra el comportamiento promedio del equipo.

**Elementos del Gráfico:**
1.  **Nodos (Círculos):** Representan a los jugadores.
    * **Posición:** Se calcula usando la **mediana** de las coordenadas de sus pases (más robusto que el promedio, ya que ignora acciones aisladas como un córner).
    * **Tamaño:** Proporcional a la cantidad de pases que inicia el jugador (su influencia en el juego).
2.  **Aristas (Líneas):** Representan los pases entre dos jugadores.
    * **Grosor:** Indica la frecuencia de conexión. Una línea gruesa entre dos centrales indica mucha circulación de seguridad; una línea gruesa hacia un delantero indica un canal de ataque principal.
3.  **Filtros:** Se eliminan conexiones débiles (menos de 3 pases) para limpiar el ruido visual y destacar solo las sociedades tácticas reales.

**Nota:** En esta versión utilizamos `metricasports` como tipo de pitch para compatibilidad con los datos de ejemplo de Metrica.

In [ ]:


def plot_pass_network_strategy(events_df, team_name, ax=None):
    # 1. Configuración del Pitch
    # Usamos 'metrica' para que mplsoccer maneje la escala 0-1 automáticamente
    if ax is None:
        pitch = Pitch(pitch_type='metricasports', pitch_length=105, pitch_width=68,
                      line_color='#c7d5cc', pitch_color='#22312b')
        fig, ax = pitch.draw(figsize=(10, 6))
    else:
        pitch = Pitch(pitch_type='metricasports', pitch_length=105, pitch_width=68,
                      line_color='#c7d5cc', pitch_color='#22312b')
        pitch.draw(ax=ax)

    # 2. Filtrar Datos: Solo pases COMPLETADOS del equipo
    passes = events_df[
        (events_df['Type'] == 'PASS') & 
        (events_df['Team'] == team_name)
    ].copy()
    
    if passes.empty:
        print(f"⚠️ No hay pases registrados para {team_name}")
        return

    # 3. Calcular Nodos (Posición Promedio - Mediana)
    # La mediana evita que un córner o saque de banda desplace la posición media del jugador
    avg_locs = passes.groupby('From').agg({
        'Start X': 'median', 
        'Start Y': 'median', 
        'Type': 'count' # Cantidad de pases (para tamaño del nodo)
    })
    avg_locs.columns = ['x', 'y', 'count']

    # 4. Calcular Aristas (Conexiones)
    passes['pair'] = passes['From'] + "_" + passes['To']
    pass_counts = passes.groupby('pair').count().reset_index()
    pass_counts = pass_counts[['pair', 'Type']].rename(columns={'Type': 'pass_count'})
    
    # 5. DIBUJAR: Conexiones (Líneas)
    MIN_PASSES = 3 # Filtro de ruido
    max_value = pass_counts['pass_count'].max()
    
    # Dibujar líneas primero (zorder bajo)
    for i, row in pass_counts.iterrows():
        try:
            player1, player2 = row['pair'].split('_')
            if player1 in avg_locs.index and player2 in avg_locs.index:
                if row['pass_count'] < MIN_PASSES: continue
                
                p1 = avg_locs.loc[player1]
                p2 = avg_locs.loc[player2]
                
                # Grosor relativo
                width = (row['pass_count'] / max_value * 5) + 0.5
                alpha = min(1, row['pass_count'] / 15) 
                
                pitch.lines(p1['x'], p1['y'], p2['x'], p2['y'],
                            lw=width, color='#00ff85', zorder=1, alpha=alpha, ax=ax)
        except: pass

    # 6. DIBUJAR: Jugadores (Nodos)
    max_node = avg_locs['count'].max()
    
    for player, row in avg_locs.iterrows():
        size = (row['count'] / max_node) * 600
        
        # Nodo
        pitch.scatter(row['x'], row['y'], s=size, 
                      c='#22312b', edgecolors='#00ff85', linewidth=2.5, zorder=2, ax=ax)
        
        # Etiqueta (Número)
        label = player.replace('Player', '').strip()
        pitch.annotate(label, xy=(row['x'], row['y']), 
                       c='white', va='center', ha='center', size=10, weight='bold', ax=ax)
        
    ax.set_title(f'ESTRUCTURA DE PASES: {team_name}', fontsize=14, color='white', pad=15)

# --- EJECUCIÓN ---
print("🕸️ Generando Análisis Macro (Conexiones)...")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.set_facecolor('#1c1c1c')

# Dibujar ambos equipos
plot_pass_network_strategy(master_table, 'Home', ax=axes[0])
plot_pass_network_strategy(master_table, 'Away', ax=axes[1])

plt.tight_layout()
plt.show()

# 15. Motor de IA Conversacional (RAG Simulado, simulador de Analista)

Esta celda implementa la lógica de un **Analista Virtual**. Utilizamos un enfoque de *RAG (Retrieval Augmented Generation)* simulado para el Hackathon, lo que nos permite ofrecer respuestas inteligentes sin depender de una API de pago externa (como GPT-4) en tiempo real.

**Arquitectura del Cerebro:**
1.  **Generador de Contexto:** Una función que agrega los KPIs de todo el partido (EPV total, altura media del bloque, velocidad) para tener una "chuleta" o resumen global siempre disponible.
2.  **Motor de Razonamiento (Rule-Based AI):** Un árbol de decisión lógica que detecta la **intención** del usuario:
    * *Intención "Pérdidas":* Si el usuario pregunta por pérdidas, el sistema cruza los datos con la **Altura del Bloque Rival**. Si el rival presionó alto (>35m), deduce que la causa fue la presión; si no, deduce error no forzado.
    * *Intención "Ataque":* Si pregunta por goles, analiza la **Velocidad media**. Si es alta, diagnostica "Contraataque"; si es baja, "Juego Posicional".
3.  **Generación de Respuesta:** Devuelve la respuesta formateada en HTML, simulando un chat real, con secciones de *Evidencia en Datos* y *Recomendación Táctica*.

In [ ]:


# 1. GENERADOR DE CONTEXTO (La "Chuleta" general)
def get_match_context_text(final_df):
    stats = final_df.groupby('Team').agg({
        'EPV_Added': 'sum',
        'Packing': 'mean',
        'Block_Height_m': 'mean',
        'Speed_ms': 'mean',
        'Phase_ID': 'count'
    }).round(2)
    
    context = "📊 <b>DATOS GENERALES DEL PARTIDO:</b><br>"
    for team in stats.index:
        s = stats.loc[team]
        context += (f"• <b>{team}</b>: {int(s['Phase_ID'])} posesiones. "
                    f"EPV Total: {s['EPV_Added']}. "
                    f"Bloque Defensivo: {s['Block_Height_m']}m. "
                    f"Velocidad: {s['Speed_ms']} m/s.<br>")
    return context

# 2. MOTOR DE RAZONAMIENTO (Simulación de IA)
def ai_tactical_analyst_logic(question, final_df):
    question = question.lower()
    
    # A. DETECCIÓN DE INTENCIÓN: "PÉRDIDAS / POSESIÓN"
    if any(x in question for x in ['pérdida', 'perdimos', 'posesión', 'recuperar']):
        # Filtramos datos reales de pérdidas
        losses = final_df[final_df['Outcome'] == 'Pérdida']
        if losses.empty: return "No detecto pérdidas significativas en los datos."
        
        # Análisis de Causa
        avg_block = losses['Block_Height_m'].mean()
        causa = "la alta presión del rival" if avg_block > 35 else "errores no forzados en construcción"
        
        respuesta = f"Analizando las {len(losses)} pérdidas de posesión registradas..."
        insight = (f"He detectado que el bloque defensivo rival estaba situado a una altura media de <b>{avg_block:.1f} metros</b> "
                   f"durante estas jugadas. Esto sugiere que {causa} fue el factor determinante.")
        recommendation = "💡 <b>Solución:</b> Buscar pases más verticales (Directness > 0.8) para saltar la línea de presión."

    # B. DETECCIÓN DE INTENCIÓN: "GOLES / ATAQUE"
    elif any(x in question for x in ['gol', 'ataque', 'marcar', 'ofensivo']):
        goals = final_df[final_df['Outcome'] == 'GOL']
        shots = final_df[final_df['Outcome'].str.contains('Tiro')]
        
        if goals.empty and shots.empty:
            return "No hay registros de goles o tiros en los datos cargados."
            
        avg_speed = goals['Speed_ms'].mean() if not goals.empty else shots['Speed_ms'].mean()
        tipo = "Contraataques Rápidos" if avg_speed > 4.0 else "Juego Posicional"
        
        respuesta = f"Revisando la producción ofensiva ({len(goals)} goles y {len(shots)} tiros)..."
        insight = (f"La velocidad media de las jugadas de peligro fue de <b>{avg_speed:.1f} m/s</b>. "
                   f"El equipo está generando peligro principalmente mediante <b>{tipo}</b>.")
        recommendation = "💡 <b>Clave:</b> Explotar los espacios a la espalda de la defensa rival."

    # C. INTENCIÓN: RESUMEN GENERAL
    else:
        respuesta = "Generando resumen ejecutivo del encuentro..."
        insight = get_match_context_text(final_df)
        recommendation = "💡 <b>Conclusión:</b> Partido definido por el control de los espacios intermedios."

    # 3. FORMATO DE RESPUESTA (HTML Chat)
    html_response = f"""
    <div style="font-family: sans-serif; max-width: 600px;">
        <div style="background-color: #f0f2f5; padding: 15px; border-radius: 15px; margin-bottom: 10px; text-align: right; color: #333;">
            <b>👤 Tú:</b> {question}
        </div>
        <div style="background-color: #e3f2fd; padding: 15px; border-radius: 15px; border-left: 5px solid #2196f3; color: #333;">
            <b>🤖 Asistente Táctico:</b><br><br>
            {respuesta}<br><br>
            <i>🔍 Evidencia en Datos:</i><br>
            {insight}<br><br>
            {recommendation}
        </div>
    </div>
    """
    return html_response

In [ ]:
ai_tactical_analyst_logic("¿Cómo podemos mejorar nuestra posesión para evitar pérdidas?", final_kpi_dataset)

In [ ]:
get_match_context_text(final_kpi_dataset)

# Interfaz de Chat (Chatbot UI, Frontend del Asistente)

Esta celda construye la **Interfaz Gráfica (GUI)** para nuestro analista virtual, cerrando el ciclo del sistema RAG.

**Componentes de la Interfaz:**
1.  **Input de Texto (`widgets.Text`):** Donde el entrenador escribe sus dudas en lenguaje natural.
2.  **Botón de Acción (`widgets.Button`):** Disparador que envía la consulta al motor lógico.
3.  **Zona de Respuesta (`widgets.Output`):** Un área dinámica que renderiza el HTML generado por la IA, permitiendo mostrar texto enriquecido (negritas, colores, iconos) en lugar de texto plano aburrido.

**Flujo:** Usuario escribe $\rightarrow$ Click $\rightarrow$ Motor Lógico (Celda anterior) $\rightarrow$ Renderizado HTML.

In [ ]:


print("💬 CHATBOT TÁCTICO (Nivel 3 - RAG Simulado)")

# Widgets
txt_chat = widgets.Text(
    placeholder='Pregunta al asistente (ej: ¿Por qué perdimos la posesión?)', 
    layout={'width': '70%'}
)
btn_send = widgets.Button(
    description='Enviar', 
    icon='paper-plane', 
    button_style='primary'
)
output_chat = widgets.Output()

# Lógica
def on_chat_send(b):
    user_q = txt_chat.value
    if not user_q: return
    
    with output_chat:
        clear_output(wait=True) # Limpiamos para efecto de "nueva respuesta"
        
        # LLAMADA AL MOTOR LÓGICO
        try:
            html_resp = ai_tactical_analyst_logic(user_q, final_kpi_dataset)
            display(HTML(html_resp))
        except NameError:
            print("❌ Error: Asegúrate de tener 'final_kpi_dataset' cargado.")
        except Exception as e:
            print(f"Error: {e}")

# Conectar
btn_send.on_click(on_chat_send)

# Mostrar
display(widgets.HBox([txt_chat, btn_send]))
display(output_chat)

al final hemos obtenido este notebook, sabiendo que esto ya es funcional, podemos añadirle los comentarios pertinenes a cada celda para entender que se hace en cada celda. 

cuando ya tengamos esto podemos crear un nuevo notebook donde al principio se defindan todos los partidos porque ahora lo hacemos con un solo partido y poder adaptar las funciones para tener contexto de mucho partidos, 